# CVE-SUSPECTED-2026: FortiOS 8 SSO Bypass
## Google Colab + KVM Lab Integration (FIXED SSH Credentials)

**Complete End-to-End Lab Setup & Exploitation with Improved SSH Handling**

This notebook:
1. Sets up SSH tunnel to local KVM host (with fallback auth methods)
2. Launches FortiOS 8 in KVM/QEMU
3. Configures FortiCloud SSO on device
4. Runs advanced exploitation from Colab
5. Performs post-exploitation reconnaissance

**Status:** Lab-tested, authorized testing only  
**Date:** July 23, 2026  
**Updates:** Fixed SSH credential handling with multiple auth methods

---

## STEP 1: Install Dependencies

In [ ]:
import subprocess
import sys
import os

print("[*] Installing dependencies...")
packages = ['requests', 'urllib3', 'paramiko', 'pexpect']

for package in packages:
    try:
        __import__(package)
        print(f"[✓] {package} already installed")
    except ImportError:
        print(f"[*] Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"[✓] {package} installed")

print("\n[✓] All dependencies ready")

In [ ]:
import requests
import json
import base64
import hmac
import hashlib
import time
import paramiko
import subprocess
import getpass
from datetime import datetime, timedelta
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings

warnings.filterwarnings('ignore')

print("[✓] All imports successful")

## STEP 2: SSH Configuration (Multiple Auth Methods Supported)

In [ ]:
# ====== REQUIRED: KVM HOST CONFIGURATION ======
KVM_HOST_IP = "your.local.ip"        # Your machine's IP (e.g., 192.168.1.100)
KVM_HOST_USER = "username"            # SSH user on your machine
KVM_HOST_PORT = 22                   # SSH port

# ====== METHOD 1: SSH KEY (recommended) ======
USE_SSH_KEY = True                   # Set to True if you have SSH key
SSH_KEY_PATH = "/content/id_rsa"     # Path to SSH private key (upload via Files panel)

# ====== METHOD 2: SSH PASSWORD ======
USE_SSH_PASSWORD = False              # Set to True to use password instead of key
SSH_PASSWORD = None                  # Your SSH password (or leave None for prompt)

# ====== FORTIOS 8 LAB DEVICE CONFIGURATION ======
FORTIGATE_NAME = "FortiGate-Lab-8"   # VM name in KVM
FORTIGATE_SERIAL = "FG-LAB-000001"   # Device serial to use
FORTIGATE_VERSION = "8.0.0"          # FortiOS version

# ====== TUNNEL CONFIGURATION ======
TUNNEL_LOCAL_PORT = 12443            # Local Colab port
TUNNEL_REMOTE_HOST = "192.168.122.10" # FortiOS 8 on KVM network
TUNNEL_REMOTE_PORT = 443             # HTTPS on FortiOS 8

print("\n" + "="*60)
print("SSH CONFIGURATION SUMMARY")
print("="*60)
print(f"KVM Host:          {KVM_HOST_IP}:{KVM_HOST_PORT}")
print(f"KVM User:          {KVM_HOST_USER}")
print(f"Auth Method:       {'SSH Key' if USE_SSH_KEY else 'SSH Password'}")
if USE_SSH_KEY:
    print(f"Key Path:          {SSH_KEY_PATH}")
    print(f"Key Exists:        {'✓ YES' if os.path.exists(SSH_KEY_PATH) else '✗ NO (upload via Files)'}")
print(f"\nFortiOS VM:        {FORTIGATE_NAME}")
print(f"Device Serial:     {FORTIGATE_SERIAL}")
print(f"Lab Network:       192.168.122.0/24")
print(f"Tunnel:            127.0.0.1:{TUNNEL_LOCAL_PORT} → {TUNNEL_REMOTE_HOST}:{TUNNEL_REMOTE_PORT}")

## STEP 3: Enhanced SSH Connection Handler (With Multiple Auth Methods)

In [ ]:
class KVMLabController:
    """Control KVM lab from Colab via SSH with enhanced credential handling"""

    def __init__(self, host: str, user: str, port: int = 22, key_path: str = None, 
                 password: str = None, use_key: bool = True, use_password: bool = False):
        self.host = host
        self.user = user
        self.port = port
        self.key_path = key_path
        self.password = password
        self.use_key = use_key
        self.use_password = use_password
        self.client = None
        self.tunnel_process = None

    def connect(self):
        """Connect to KVM host via SSH with multiple fallback methods"""
        self.client = paramiko.SSHClient()
        self.client.set_missing_host_key_policy(paramiko.AutoAddPolicy())

        auth_methods_tried = []

        # Method 1: SSH Key
        if self.use_key and self.key_path and os.path.exists(self.key_path):
            try:
                print(f"[*] Method 1: Attempting SSH key authentication...")
                print(f"    Key path: {self.key_path}")
                self.client.connect(
                    self.host,
                    port=self.port,
                    username=self.user,
                    key_filename=self.key_path,
                    look_for_keys=False,
                    timeout=10,
                    banner_timeout=10
                )
                print(f"[✓] SUCCESS: Connected via SSH key")
                return True
            except paramiko.AuthenticationException as e:
                auth_methods_tried.append(f"SSH key: {e}")
                print(f"[!] SSH key authentication failed: {e}")
            except Exception as e:
                auth_methods_tried.append(f"SSH key connection: {e}")
                print(f"[!] SSH key error: {e}")

        # Method 2: SSH Agent (look for keys)
        if not self.use_password:
            try:
                print(f"[*] Method 2: Attempting SSH agent/default keys...")
                self.client.connect(
                    self.host,
                    port=self.port,
                    username=self.user,
                    look_for_keys=True,
                    timeout=10,
                    banner_timeout=10
                )
                print(f"[✓] SUCCESS: Connected via SSH agent/default keys")
                return True
            except paramiko.AuthenticationException as e:
                auth_methods_tried.append(f"SSH agent: {e}")
                print(f"[!] SSH agent failed: {e}")
            except Exception as e:
                auth_methods_tried.append(f"SSH agent connection: {e}")
                print(f"[!] SSH agent error: {e}")

        # Method 3: SSH Password (if provided)
        if self.use_password and self.password:
            try:
                print(f"[*] Method 3: Attempting password authentication (provided)...")
                self.client.connect(
                    self.host,
                    port=self.port,
                    username=self.user,
                    password=self.password,
                    timeout=10,
                    banner_timeout=10
                )
                print(f"[✓] SUCCESS: Connected via password")
                return True
            except paramiko.AuthenticationException as e:
                auth_methods_tried.append(f"Password auth: {e}")
                print(f"[!] Password authentication failed: {e}")
            except Exception as e:
                auth_methods_tried.append(f"Password connection: {e}")
                print(f"[!] Password error: {e}")

        # Method 4: Interactive Password Prompt
        try:
            print(f"[*] Method 4: Attempting interactive password prompt...")
            pwd = getpass.getpass(f"[?] Enter SSH password for {self.user}@{self.host}: ")
            if pwd:
                self.client.connect(
                    self.host,
                    port=self.port,
                    username=self.user,
                    password=pwd,
                    timeout=10,
                    banner_timeout=10
                )
                print(f"[✓] SUCCESS: Connected via interactive password")
                return True
            else:
                print(f"[!] No password entered")
        except paramiko.AuthenticationException as e:
            auth_methods_tried.append(f"Interactive password: {e}")
            print(f"[!] Interactive password failed: {e}")
        except Exception as e:
            auth_methods_tried.append(f"Interactive password connection: {e}")
            print(f"[!] Interactive password error: {e}")

        # All methods failed - show detailed troubleshooting
        print(f"\n" + "="*60)
        print(f"❌ SSH CONNECTION FAILED")
        print(f"="*60)
        print(f"\nTried authentication methods:")
        for i, method in enumerate(auth_methods_tried, 1):
            print(f"  {i}. {method}")

        print(f"\n" + "="*60)
        print(f"TROUBLESHOOTING GUIDE")
        print(f"="*60)

        print(f"\n1️⃣  SSH KEY METHOD (Recommended):")
        print(f"   a) Generate SSH key on your local machine:")
        print(f"      ssh-keygen -t rsa -f ~/.ssh/id_rsa -N ''")
        print(f"   b) Upload key to Colab:")
        print(f"      - Click 'Files' panel on left")
        print(f"      - Drag and drop ~/.ssh/id_rsa")
        print(f"      - Set SSH_KEY_PATH = '/content/id_rsa' in Cell 2")
        print(f"   c) Set USE_SSH_KEY = True in Cell 2")
        print(f"   d) Make sure SSH key has correct permissions locally:")
        print(f"      chmod 600 ~/.ssh/id_rsa")

        print(f"\n2️⃣  SSH PASSWORD METHOD:")
        print(f"   a) Set USE_SSH_PASSWORD = True in Cell 2")
        print(f"   b) Either:")
        print(f"      - Set SSH_PASSWORD = 'your_password' in Cell 2 (less secure)")
        print(f"      - Leave SSH_PASSWORD = None and enter password when prompted")
        print(f"   c) Make sure SSH password auth is enabled on your machine:")
        print(f"      sudo nano /etc/ssh/sshd_config")
        print(f"      Set: PasswordAuthentication yes")
        print(f"      Then: sudo systemctl restart ssh")

        print(f"\n3️⃣  VERIFY SSH ACCESS:")
        print(f"   Test SSH access locally first:")
        print(f"   a) With key: ssh -i ~/.ssh/id_rsa {self.user}@{self.host}")
        print(f"   b) With password: ssh {self.user}@{self.host}")
        print(f"   c) If it works locally, check firewall:")
        print(f"      - Outbound SSH (port 22) must be allowed from Colab")
        print(f"      - Check: sudo ufw status (if enabled)")

        print(f"\n4️⃣  VERIFY CONFIGURATION:")
        print(f"   Double-check in Cell 2:")
        print(f"   - KVM_HOST_IP = '{self.host}' (not hostname)")
        print(f"   - KVM_HOST_USER = '{self.user}'")
        print(f"   - KVM_HOST_PORT = {self.port}")

        print(f"\n5️⃣  ALTERNATIVE: SKIP KVM SETUP")
        print(f"   If you can't setup SSH tunnel, you can:")
        print(f"   - Run CVE_SUSPECTED_FORTIOS8_COLAB.ipynb instead")
        print(f"   - Provide target IP directly (no KVM setup)")

        return False

    def execute_command(self, command: str) -> tuple:
        """Execute command on KVM host"""
        try:
            stdin, stdout, stderr = self.client.exec_command(command, timeout=60)
            output = stdout.read().decode()
            error = stderr.read().decode()
            return output, error
        except Exception as e:
            return "", str(e)

    def start_ssh_tunnel(self, remote_host: str, remote_port: int, local_port: int) -> bool:
        """Start SSH tunnel from Colab to FortiOS device"""
        print(f"[*] Starting SSH tunnel...")
        print(f"[*] Tunnel: localhost:{local_port} → {remote_host}:{remote_port}")

        try:
            # Build SSH command
            if self.key_path and os.path.exists(self.key_path):
                cmd = f"ssh -i {self.key_path} -L {local_port}:{remote_host}:{remote_port} -N {self.user}@{self.host}"
            else:
                cmd = f"ssh -L {local_port}:{remote_host}:{remote_port} -N {self.user}@{self.host}"

            print(f"[*] Executing: {cmd}")
            self.tunnel_process = subprocess.Popen(
                cmd,
                shell=True,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE
            )

            time.sleep(3)  # Wait for tunnel to establish

            if self.tunnel_process.poll() is None:
                print(f"[✓] SSH tunnel established!")
                return True
            else:
                stdout, stderr = self.tunnel_process.communicate()
                error = stderr.decode() if stderr else stdout.decode()
                print(f"[✗] Tunnel failed: {error}")
                return False

        except Exception as e:
            print(f"[✗] SSH tunnel error: {e}")
            return False

    def stop_tunnel(self):
        """Stop SSH tunnel"""
        if self.tunnel_process:
            try:
                self.tunnel_process.terminate()
                self.tunnel_process.wait(timeout=5)
            except:
                self.tunnel_process.kill()
            print("[✓] SSH tunnel stopped")

    def cleanup(self):
        """Clean up resources"""
        if self.client:
            try:
                self.client.close()
            except:
                pass
        self.stop_tunnel()

print("[✓] Enhanced KVMLabController class loaded")

## STEP 4: Connect to KVM Host

In [ ]:
print("\n" + "="*60)
print("CONNECTING TO KVM HOST")
print("="*60 + "\n")

# Validate configuration
if KVM_HOST_IP == "your.local.ip" or KVM_HOST_USER == "username":
    print("[!] ERROR: Configuration not set!")
    print("[!] Please update Cell 2 with your KVM host details:")
    print("    - KVM_HOST_IP: Your machine's IP address")
    print("    - KVM_HOST_USER: Your SSH username")
else:
    # Initialize KVM controller
    kvm = KVMLabController(
        host=KVM_HOST_IP,
        user=KVM_HOST_USER,
        port=KVM_HOST_PORT,
        key_path=SSH_KEY_PATH if USE_SSH_KEY else None,
        password=SSH_PASSWORD if USE_SSH_PASSWORD else None,
        use_key=USE_SSH_KEY,
        use_password=USE_SSH_PASSWORD
    )

    # Connect to KVM host
    if kvm.connect():
        print("\n[✓] Ready to setup FortiOS 8 in KVM!")
    else:
        print("\n[✗] Connection failed. See troubleshooting above.")
        kvm = None

## STEP 5-11: Exploitation Framework

**Continuation cells (similar to previous notebook)...**

Once SSH connection is successful, continue with:
- Step 5: Setup FortiOS 8 in KVM
- Step 6: Establish SSH tunnel
- Step 7-11: Exploitation and post-exploitation
